In [1]:
import pandas as pd
import sqlite3

## Что нужно сделать:

1. Создай соединение с базой данных с помощью библиотеки sqlite3.
2. Одним запросом создай в базе новую таблицу datamart, объединив таблицы pageviews и checker.
    - В таблице должны быть колонки: "uid", "labname", "first_commit_ts", "first_view_ts".
    - "first_commit_ts" - новое имя колонки "timestamp" из таблицы checker; это первая отправка для конкретной лабораторной и пользователя.
    - "first_view_ts" - первая временная метка посещения пользователем ленты (newsfeed) из pageviews.
    - Фильтр status = 'ready' должен сохраняться.
    - Фильтр numTrials = 1 должен сохраняться.
    - "labnames" должен входить в список: laba04, laba04s, laba05, laba06, laba06s, и project1.
    - В таблицу должны попадать только пользователи (uid вида user_*), администраторы - исключаются.
    - Колонки "first_commit_ts" и "first_view_ts" должны иметь тип datetime64[ns].
3. С помощью методов Pandas создай два DataFrame: test и control.
    - test содержит пользователей с заполненными значениями в "first_view_ts".
    - control содержит пользователей с пропущенными значениями в "first_view_ts".
    - Замени пропуски в control средним значением "first_view_ts" среди пользователей из test. Сохрани обе таблицы в базе - они понадобятся в следующих упражнениях.
4. Закрой соединение.

In [2]:
db_path = '../data/checking-logs.sqlite'
conn = sqlite3.connect(db_path)

In [3]:
conn.execute("DROP TABLE IF EXISTS datamart;")

In [4]:
conn.execute(
    """
    CREATE TABLE IF NOT EXISTS datamart AS
    SELECT
        c.uid,
        c.labname,
        c.timestamp AS first_commit_ts,
        MIN(p.datetime) AS first_view_ts
    FROM checker c
    LEFT JOIN pageviews p ON c.uid = p.uid
    WHERE c.status = 'ready'
      AND c.numTrials = 1
      AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
      AND c.uid LIKE 'user_%'
    GROUP BY c.uid, c.labname, c.timestamp;
    """
)

datamart = pd.io.sql.read_sql(
    'SELECT * FROM datamart;',
    conn,
    parse_dates = ["first_commit_ts", "first_view_ts"]
)
datamart[["first_commit_ts", "first_view_ts"]] = datamart[["first_commit_ts", "first_view_ts"]].astype("datetime64[ns]")

In [5]:
print('===Датафрейма datamart===')
datamart

===Датафрейма datamart===


,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
...,...,...,...,...
135,user_8,laba04s,2020-04-19 10:22:35.761944,NaT
136,user_8,laba05,2020-05-02 13:28:07.705193,NaT
137,user_8,laba06,2020-05-16 17:56:15.755553,NaT
138,user_8,laba06s,2020-05-16 20:01:07.900727,NaT


In [6]:
datamart.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    str           
 1   labname          140 non-null    str           
 2   first_commit_ts  140 non-null    datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), str(2)
memory usage: 4.5 KB


In [7]:
print(f'Тип колонок с датами:\n{datamart[['first_commit_ts', 'first_view_ts']].dtypes}')

Тип колонок с датами:
first_commit_ts    datetime64[ns]
first_view_ts      datetime64[ns]
dtype: object


In [8]:
test = datamart[datamart['first_view_ts'].notna()].copy()
control = datamart[datamart['first_view_ts'].isna()].copy()

In [9]:
print('===Датафрейм test===')
test

===Датафрейм test===


,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


In [10]:
print('===Датафрейм control (до)===')
control

===Датафрейм control (до)===


,uid,labname,first_commit_ts,first_view_ts
12,user_11,laba05,2020-05-03 21:06:55.970293,NaT
13,user_11,project1,2020-05-03 23:45:33.673409,NaT
14,user_12,laba04,2020-04-18 17:07:51.767358,NaT
15,user_12,laba04s,2020-04-26 15:42:38.070593,NaT
16,user_12,laba05,2020-05-03 08:39:25.174316,NaT
...,...,...,...,...
135,user_8,laba04s,2020-04-19 10:22:35.761944,NaT
136,user_8,laba05,2020-05-02 13:28:07.705193,NaT
137,user_8,laba06,2020-05-16 17:56:15.755553,NaT
138,user_8,laba06s,2020-05-16 20:01:07.900727,NaT


In [11]:
test.info()

<class 'pandas.DataFrame'>
Index: 59 entries, 0 to 114
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     str           
 1   labname          59 non-null     str           
 2   first_commit_ts  59 non-null     datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), str(2)
memory usage: 2.3 KB


In [12]:
control.info()

<class 'pandas.DataFrame'>
Index: 81 entries, 12 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     str           
 1   labname          81 non-null     str           
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    0 non-null      datetime64[ns]
dtypes: datetime64[ns](2), str(2)
memory usage: 3.2 KB


In [13]:
mean_first_view_ts = test['first_view_ts'].mean()
control['first_view_ts'] = mean_first_view_ts

In [14]:
print('===Датафрейм control (после)===')
control

===Датафрейм control (после)===


,uid,labname,first_commit_ts,first_view_ts
12,user_11,laba05,2020-05-03 21:06:55.970293,2020-04-27 00:40:05.761783552
13,user_11,project1,2020-05-03 23:45:33.673409,2020-04-27 00:40:05.761783552
14,user_12,laba04,2020-04-18 17:07:51.767358,2020-04-27 00:40:05.761783552
15,user_12,laba04s,2020-04-26 15:42:38.070593,2020-04-27 00:40:05.761783552
16,user_12,laba05,2020-05-03 08:39:25.174316,2020-04-27 00:40:05.761783552
...,...,...,...,...
135,user_8,laba04s,2020-04-19 10:22:35.761944,2020-04-27 00:40:05.761783552
136,user_8,laba05,2020-05-02 13:28:07.705193,2020-04-27 00:40:05.761783552
137,user_8,laba06,2020-05-16 17:56:15.755553,2020-04-27 00:40:05.761783552
138,user_8,laba06s,2020-05-16 20:01:07.900727,2020-04-27 00:40:05.761783552


In [18]:
test.to_sql('test', conn, if_exists='replace', index=False)

ProgrammingError: Cannot operate on a closed database.

In [19]:
control.to_sql('control', conn, if_exists='replace', index=False)

ProgrammingError: Cannot operate on a closed database.

In [17]:
conn.close()

In [20]:
db_path = '../data/checking-logs.sqlite'
conn = sqlite3.connect(db_path)

In [21]:
test = pd.io.sql.read_sql('SELECT * FROM test', conn)

In [22]:
control = pd.io.sql.read_sql('SELECT * FROM control', conn)

In [23]:
print(test)

        uid   labname             first_commit_ts               first_view_ts
0    user_1    laba04  2020-04-26 17:06:18.462708  2020-04-26 21:53:59.624136
1    user_1   laba04s  2020-04-26 17:12:11.843671  2020-04-26 21:53:59.624136
2    user_1    laba05  2020-05-02 19:15:18.540185  2020-04-26 21:53:59.624136
3    user_1    laba06  2020-05-17 16:26:35.268534  2020-04-26 21:53:59.624136
4    user_1   laba06s  2020-05-20 12:23:37.289724  2020-04-26 21:53:59.624136
5    user_1  project1  2020-05-14 20:56:08.898880  2020-04-26 21:53:59.624136
6   user_10    laba04  2020-04-25 08:24:52.696624  2020-04-18 12:19:50.182714
7   user_10   laba04s  2020-04-25 08:37:54.604222  2020-04-18 12:19:50.182714
8   user_10    laba05  2020-05-01 19:27:26.063245  2020-04-18 12:19:50.182714
9   user_10    laba06  2020-05-19 11:39:28.885637  2020-04-18 12:19:50.182714
10  user_10   laba06s  2020-05-20 07:37:31.175817  2020-04-18 12:19:50.182714
11  user_10  project1  2020-05-12 20:12:28.056618  2020-04-18 12

In [24]:
print(control)

        uid   labname             first_commit_ts               first_view_ts
0   user_11    laba05  2020-05-03 21:06:55.970293  2020-04-27 00:40:05.761783
1   user_11  project1  2020-05-03 23:45:33.673409  2020-04-27 00:40:05.761783
2   user_12    laba04  2020-04-18 17:07:51.767358  2020-04-27 00:40:05.761783
3   user_12   laba04s  2020-04-26 15:42:38.070593  2020-04-27 00:40:05.761783
4   user_12    laba05  2020-05-03 08:39:25.174316  2020-04-27 00:40:05.761783
..      ...       ...                         ...                         ...
76   user_8   laba04s  2020-04-19 10:22:35.761944  2020-04-27 00:40:05.761783
77   user_8    laba05  2020-05-02 13:28:07.705193  2020-04-27 00:40:05.761783
78   user_8    laba06  2020-05-16 17:56:15.755553  2020-04-27 00:40:05.761783
79   user_8   laba06s  2020-05-16 20:01:07.900727  2020-04-27 00:40:05.761783
80   user_8  project1  2020-05-14 15:42:04.002981  2020-04-27 00:40:05.761783

[81 rows x 4 columns]
